In [ ]:
# 1. Install Python 3.11 environment in Colab
!apt-get update -y && apt-get install python3.11 python3.11-distutils -y
!curl -sS https://bootstrap.pypa.io/get-pip.py | python3.11

# 2. Install dependencies into python3.11
!python3.11 -m pip install akida==2.16.1 cnn2snn==2.16.1 quantizeml==0.19.0 akida-models==1.10.0 tensorflow==2.15.0 tensorflow-datasets importlib_resources matplotlib_inline IPython 

# 3. Run test_fbz
!python3.11 -m src.test_fbz --model Synapse/subsystems/payload/models/baseline_v1/model.fbz


In [12]:
!python3.11 -m src.test_fbz --model src/data/models/baseline_model.fbz


2026-08-22 05:20:38.129634: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-22 05:20:38.263391: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-08-22 05:20:38.263473: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-08-22 05:20:38.266715: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-22 05:20:38.286063: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-22 05:20:38.286434: I tensorflow/core/platform/cpu_feature_guard.cc:1

In [13]:
!python3.11 -m src.test_fbz --model src/data/models/attuned_model.fbz


2026-08-22 05:27:02.335426: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-22 05:27:02.422594: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2026-08-22 05:27:02.422673: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2026-08-22 05:27:02.424769: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-08-22 05:27:02.438916: I external/local_tsl/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-08-22 05:27:02.439327: I tensorflow/core/platform/cpu_feature_guard.cc:1

In [ ]:
def extreme_values(image):
    """
    This function returns False if more than 30% of pixels are in the high extreme [230, 255] or low extreme [0, 20], or True if the pixels are in a normal range.
    """
    np_image = np.array(image.convert("L"))
    pct = np.mean((np_image < 20) | (np_image > 230)) # percentage of extreme pixels

    return pct <= 0.30

In [ ]:
def cloud_deck(image):
    """
    This function returns False if the image is both excessively bright and last contrast spread, or True otherwise.
    """
    np_image = np.array(image.convert("L"))
    gray_mean = np.mean(np_image)
    gray_std = np.std(np_image)

    return gray_mean <= 170 or gray_std >= 15

In [ ]:
def check_blur(image):
    """Returns True if the image is in focus or is valid water, False if blurred"""
    np_image = np.array(image.convert("L"))
    gray_mean = np.mean(np_image)

    if gray_mean < 60:
        return True

    # laplacian convolution
    laplacian = (
        np_image[:-2, 1:-1] # top
        + np_image[2:, 1:-1] # bottom
        + np_image[1:-1, :-2] # left
        + np_image[1:-1, 2:] # right
        - 4 * np_image[1:-1, 1:-1] # center
    )

    laplacian_var = np.var(laplacian)

    return laplacian_var > 30.0

In [ ]:
def check_haze(image):
    """Return True if clear, False if washed by atmospheric haze."""
    np_image = np.array(image.convert("HSV"))

    sat_mean = np.mean(np_image[:, :, 1])
    val_mean = np.mean(np_image[:, :, 2])

    is_hazy = (val_mean > 140) and (sat_mean < 35)
    return not is_hazy